In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
from numpy import shape
class MultiHeadAttention(nn.Module):
    def __inti__(self, num_heads:int, d_model:int):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = self.d_model // self.num_heads

        # W_Q, W_K = shape(batch_size, d_model, d_k)
        # W_V = shape(batch_size, d_model, d_v)
        self.W_Q = nn.Linear(self.d_model, self.d_model, bias = False)
        self.W_K = nn.Linear(self.d_model, self.d_model, bias = False)
        self.W_V = nn.Linear(self.d_model, self.d_model, bias = False)

        # W_O = shape(batch_size, d_model, d_model)
        self.W_O = nn.Linear(self.d_model, self.d_model, bias = False)

    def scaled_dot_product_attention(self, Q, K, V, mask = None):
        """
        Args:
            Q: shape (batch_size, seq_len_q, d_k)
            K: shape (batch_size, seq_len_k, d_k)
            V: shape (batch_size, seq_len_k, d_k)
            mask (torch.Tensor, optional): Mask tensor to prevent attention to certain positions.

        Returns:
            attn_probs: shape(batch_size, seq_len_q, seq_len_k)
            attn_output: shape(batch_size, seq_len_q, d_k)
        """

        attn_scores = torch.matmul(Q, K.transpose(-1,-2))/math.sqrt(self.d_k)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        attn_probs = F.softmax(attn_scores)

        attn_output = torch.matmul(attn_probs,V)

        return attn_output, attn_probs

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            query: shape (batch_size, seq_len_q, d_model)
            key: shape (batch_size, seq_len_k, d_model)
            value: shape (batch_size, seq_len_k, d_model)
            mask (torch.Tensor, optional): Mask tensor to prevent attention to certain positions.

        Returns:
            output: shape(batch_size, seq_len_q, d_model)
        """

        batch_size = query.size(0)

        # applying linear projection to the query, key and value, reshaping Q,K,V to separate the heads
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # reshape from (batch_size, seq_len_q, d_model) -> (batch_size, num_heads, seq_len_q, d_k)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)

        attn_output, attn_probs = self.scaled_dot_product_attention(Q,K,V,mask)
        # attn_probs: shape(batch_size, num_heads, seq_len_q, seq_len_k)
        # attn_output: shape(batch_size, num_heads, seq_len_q, d_k)

        
        # concatenate heads by reshaping attn_output to shape(batch_size, seq_len_q, num_heads, d_k) 
        # and then using view
        attn_output = attn_output.transpose(1,2).contiguous().view(batch_size, -1, self.d_model)

        # applying linear projection to the final output
        # Input shape: (batch_size, seq_len_q, d_model)
        # Output shape: (batch_size, seq_len_q, d_model)
        output = self.W_O(attn_output)

        return output



